In [1]:
#
# assure le reload de src si modifications sont faite
#
%load_ext autoreload
%autoreload 2

In [2]:
#
# import utilitaires
#
%matplotlib inline

import math
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

from pathlib import Path

In [3]:
#
# import package develope pour le projet
#
import ffury
from ffury.configs import load_config, DEFAULT_CONFIG_FILE

def get_config():
    # creation config - on sait que DEFAULT_CONFIG_FILE est dans le repertoire parent
    config = Path(ffury.__file__).parents[2].joinpath(DEFAULT_CONFIG_FILE)
    return load_config(config)

config = get_config()

# load dataset BirdCLEF
train_df = pd.read_csv(config.get_dataset_train_filename())

display(train_df.head())
print(train_df.shape)

,common_name,primary_label,latitude,longitude,filename
0,Common Sandpiper,comsan,52.0567,1.1482,comsan/XC582037.ogg
1,African Emerald Cuckoo,afecuc1,-1.0006,29.6189,afecuc1/XC700647.ogg
2,White-browed Robin-Chat,wbrcha2,-19.1046,32.7532,wbrcha2/XC522668.ogg
3,Common Sandpiper,comsan,43.1896,-0.4409,comsan/XC665470.ogg
4,Amethyst Sunbird,amesun2,-33.3214,26.6122,amesun2/XC438652.ogg


(8905, 5)


In [21]:
species_lookup = train_df[["primary_label", "common_name"]].groupby("primary_label").first()
species_lookup.reset_index(inplace=True)

num_species = species_lookup.shape[0]
eye = np.eye(num_species, dtype=np.uint8)
species_lookup["ohe"] = [eye[i]   for i in range(num_species)]

indices = pd.Categorical(train_df["primary_label"], categories=species_lookup["primary_label"])

if "primary_label_index" in train_df.columns:
    train_df.drop(columns=["primary_label_index"], inplace=True)

train_df.insert(loc=2, column="primary_label_index", value=indices.codes)


display(names_lookup.head())
print(names_lookup.shape)

,primary_label,common_name,ohe
0,abethr1,African Bare-eyed Thrush,"[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1,abhori1,African Black-headed Oriole,"[0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
2,abythr1,Abyssinian Thrush,"[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
3,afbfly1,African Blue Flycatcher,"[0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
4,afdfly1,African Dusky Flycatcher,"[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


(212, 3)


In [22]:
display(train_df.head())
print(train_df.shape)

for _, r in train_df.iterrows():
    names = names_lookup.loc[ r["primary_label_index"] ]
    assert r["common_name"] == names["common_name"] and r["primary_label"] == names["primary_label"]

,common_name,primary_label,primary_label_index,latitude,longitude,filename
0,Common Sandpiper,comsan,62,52.0567,1.1482,comsan/XC582037.ogg
1,African Emerald Cuckoo,afecuc1,5,-1.0006,29.6189,afecuc1/XC700647.ogg
2,White-browed Robin-Chat,wbrcha2,185,-19.1046,32.7532,wbrcha2/XC522668.ogg
3,Common Sandpiper,comsan,62,43.1896,-0.4409,comsan/XC665470.ogg
4,Amethyst Sunbird,amesun2,15,-33.3214,26.6122,amesun2/XC438652.ogg


(8905, 6)
